## Notebook set-up
---

Run the following cells to set up notebook for generation. These cells install dependencies and downloads Qwen for use




In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [3]:
!ollama pull qwen3

In [4]:
!ollama ps

NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


In [ ]:
!pip install langchain-ollama

In [5]:
import ollama
import json, re

def generate_bio_ollama(prompt):
    response = ollama.chat(
        model='qwen3',
        think=True,  # for enabling thinking mode
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'temperature': 0,
            'num_predict': 2048,
        }
    )

    content = response['message']['content']
    thinking = response['message'].get('thinking', '')

    print("THINKING:\n", thinking)
    print("\nBIO:\n", content)
    print(f"Total generated tokens (exact): {response['eval_count']}")

In [ ]:
!pip install wikipedia-api

import wikipediaapi

In [7]:
# Fetching wiki page
def get_wiki_context(entity_name, lang="en", sentences=5):
    wiki = wikipediaapi.Wikipedia(language=lang, user_agent="research-bot/1.0")
    page = wiki.page(entity_name)
    if not page.exists():
        print(f"[WARN] No Wikipedia page found for '{entity_name}' in lang='{lang}'")
        return None
    return ". ".join(page.summary.split(". ")[:sentences])

# RAG generation
def generate_bio_with_rag(prompt, entity_name, lang="en", **kwargs):
    context = get_wiki_context(entity_name, lang=lang)
    if context:
        augmented_prompt = f"Use the following factual context:\n{context}\n\n{prompt}"
    else:
        augmented_prompt = prompt

    print("=" * 60)
    print("AUGMENTED PROMPT:")
    print("=" * 60)
    print(augmented_prompt)
    print("=" * 60 + "\n")

    return generate_bio_ollama(augmented_prompt, **kwargs)

# **Baseline Generations**
### Biographies for example leader (Xi Jinping) in each language

In [ ]:
# English
en_bio = generate_bio_ollama("Write a biography of Xi Jinping.")
print("ENGLISH:\n", en_bio)

In [ ]:
# Hindi
hi_bio = generate_bio_ollama("शी जिनपिंग की जीवनी लिखें।")

In [ ]:
# Bengali
bn_bio = generate_bio_ollama("শি জিনপিং-এর জীবনী লিখুন।")

In [ ]:
# Swahili
sw_bio = generate_bio_ollama("Andika wasifu wa Xi Jinping.")

# **RAG Generations**
### Biographies using English & Native Wikipedia retrieval

In [ ]:
# English RAG generation
en_bio_rag = generate_bio_with_rag(
    "Write a biography of Xi Jinping.",
    entity_name="Xi Jinping",
    lang="en"
)

In [ ]:
# Hindi w/ English RAG
hi_bio_rag_en = generate_bio_with_rag(
    "शी जिनपिंग की जीवनी लिखें।",
    entity_name="Xi Jinping",
    lang="en"
)

# Hindi w/ Native RAG
hi_bio_rag_native = generate_bio_with_rag(
    "शी जिनपिंग की जीवनी लिखें।",
    entity_name="शी जिनपिंग",
    lang="hi"
)

In [ ]:
# Bengali w/ English RAG
bn_bio_rag_en = generate_bio_with_rag(
    "শি জিনপিংয়ের জীবনী লিখুন।",
    entity_name="Xi Jinping",
    lang="en"
)

# Bengali w/ Native RAG
bn_bio_rag_native = generate_bio_with_rag(
    "শি জিনপিংয়ের জীবনী লিখুন।",
    entity_name="শি জিনপিং",
    lang="bn"
)

In [ ]:
# Swahili w/ English RAG
sw_bio_rag_en = generate_bio_with_rag(
    "Andika wasifu wa Xi Jinping.",
    entity_name="Xi Jinping",
    lang="en"
)

# Swahili w/ Native RAG
sw_bio_rag_native = generate_bio_with_rag(
    "Andika wasifu wa Xi Jinping.",
    entity_name="Xi Jinping",
    lang="sw"
)

# **CoT Generations**
### Biographies using zero-shot (Version 1) & plan-and-solve prompting (Version 2)

In [ ]:
# English
# Version 1
en_bio_cot1 = generate_bio_ollama("Question: Write a biography of Xi Jinping. Answer: Let's think about it step-by-step.")

# Version 2
en_bio_cot2 = generate_bio_ollama("Question: Write a biography of Xi Jinping.\n" +
"Answer: Let's think step by step to ensure all dates and roles are 100% factual.\n" +
"1. Birth details:\n" +
"2. Education:\n" +
"3. Major Political Roles:\n" +
"4. Final Biography:")

In [ ]:
# Hindi
# Version 1
hi_bio_cot1 = generate_bio_ollama("प्रश्न: शी जिनपिंग की जीवनी लिखें। उत्तर: चलिए एक-एक कदम करके सोचते हैं।")

# Version 2
hi_bio_cot2 = generate_bio_ollama("प्रश्न: शी जिनपिंग की जीवनी लिखें।\n" +
"उत्तर: आइए चरण-दर-चरण सोचें ताकि सभी तिथियाँ और भूमिकाएँ 100% तथ्यात्मक हों।\n" +
"1. जन्म विवरण::\n" +
"2. शिक्षा:\n" +
"3. प्रमुख राजनीतिक भूमिकाएँ:\n" +
"4. Fअंतिम जीवनी:")

In [ ]:
# Bengali
# Version 1
bn_bio_cot1 = generate_bio_ollama("প্রশ্ন: শি জিনপিং-এর জীবনী লিখুন।   উত্তর: চলুন ধাপে ধাপে ভাবি।")

# Version 2
bn_bio_cot2 = generate_bio_ollama("প্রশ্ন: শি জিনপিং-এর জীবনী লিখুন।\n" +
"উত্তর: সব তারিখ ও ভূমিকা ১০০% সঠিক আছে কি না নিশ্চিত করতে ধাপে ধাপে ভাবা যাক।\n" +
"১. জন্ম সংক্রান্ত বিবরণ:\n" +
"২. শিক্ষা:\n" +
"৩. প্রধান রাজনৈতিক ভূমিকা:\n" +
"৪. চূড়ান্ত জীবনী:")

In [ ]:
# Swahili
# Version 1
sw_bio_cot1 = generate_bio_ollama("Swali: Andika wasifu wa Xi Jinping.   Jibu: Tufikirie hatua kwa hatua.")

# Version 2
sw_bio_cot2 = generate_bio_ollama("Swali: Andika wasifu wa Xi Jinping.\n" +
"Jibu: Tufikirie hatua kwa hatua ili kuhakikisha tarehe na nyadhifa zote ni sahihi kwa asilimia 100.\n" +
"1. Maelezo ya kuzaliwa:\n" +
"2. Elimu:\n" +
"3. Nyadhifa Kuu za Kisiasa:\n" +
"4. Wasifu wa Mwisho:")